
# MarianMT (zh->en) — Baseline TF Training (Without Wuxia Domain)

**TFG Hugo Silvosa– Baseline NMT (MarianMT)**  
Este cuaderno evalúa un modelo **MarianMT** (`Helsinki-NLP/opus-mt-zh-en`) sin entrenar en dominio usando un dataset de **wuxia** (chino->inglés) ya preparado en formato `datasets` (HF).





## 1) Entorno de ejecución e instalación de dependencias

In [1]:

import os, random, math
import numpy as np

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Número de GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))



CUDA disponible: True
Número de GPUs: 1
Nombre de la GPU: NVIDIA GeForce RTX 3060



> **Requisitos del dataset**: directorio HF Datasets con *splits* `train`, `validation`, `test` y columnas `zh` (chino) y `en` (inglés):  
> `processed_data/wuxia_zh_en_clean/`

In [2]:
# Configuración de carpetas para entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent.parent
BASE_DIR.mkdir(exist_ok=True)

for sub in ["evaluation", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Estructura creada (si no existía):")
for p in ["evaluation", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())

# Nota: el dataset debe existir en: CORPUS/proccesed data/wuxia_zh_en_clean


Base: C:\Users\Usuario\Desktop\TFG\CORPUS
Estructura creada (si no existía):
 - C:\Users\Usuario\Desktop\TFG\CORPUS\evaluation
 - C:\Users\Usuario\Desktop\TFG\CORPUS\models
 - C:\Users\Usuario\Desktop\TFG\CORPUS\proccesed data


## 2) Configuración

In [3]:
from dataclasses import dataclass

@dataclass
class Config:
    # Rutas (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"   # <- carpeta con dataset HuggingFace guardado con load_from_disk
    output_dir: Path   = BASE_DIR / "models" / "marianmt_wuxia_pretrain"              # <- aquí se guardarán los runs/modelos
    ckpt_dir: Path     = BASE_DIR / "checkpoints"                          # <- checkpoints durante entrenamiento
    training_dir: Path = BASE_DIR / "training"         
    evaluation_dir: Path = BASE_DIR / "evaluation"
    translate_dir: Path = BASE_DIR / "evaluation" / "translate"
    translate_file: Path =   "pre_marianmt.txt"
    results_file: Path = "pre_results.txt"
    # Columnas del dataset
    src_col: str = "zh"
    tgt_col: str = "en"

    # Modelo
    model_ckpt: str = "Helsinki-NLP/opus-mt-zh-en"

    # Entrenamiento
    seed: int = 42
    max_source_length: int = 128
    max_target_length: int = 128
    batch_size: int = 16
    epochs: int = 10
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 3

    fraction: float = 1

cfg = Config()
print(cfg)


Config(dataset_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/processed_data/wuxia_zh_en_clean'), output_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/models/marianmt_wuxia_pretrain'), ckpt_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/checkpoints'), training_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/training'), evaluation_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/evaluation'), translate_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/evaluation/translate'), translate_file='pre_marianmt.txt', results_file='pre_results.txt', src_col='zh', tgt_col='en', model_ckpt='Helsinki-NLP/opus-mt-zh-en', seed=42, max_source_length=128, max_target_length=128, batch_size=16, epochs=10, learning_rate=2e-05, weight_decay=0.01, early_stopping_patience=3, fraction=1)


In [4]:
import random, numpy as np, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)



# Semillas para reproducibilidad
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)


if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    # Para reproducibilidad estricta 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Semillas fijadas y backend configurado.")


Usando dispositivo: cuda
Semillas fijadas y backend configurado.


## 3) Cargar dataset (Hugging Face Datasets)

In [5]:

from datasets import load_from_disk, DatasetDict

assert os.path.isdir(cfg.dataset_dir), f"No se encuentra el dataset en: {cfg.dataset_dir}"
raw_ds: DatasetDict = load_from_disk(cfg.dataset_dir)
print(raw_ds)

# Validar columnas
def _check_cols(ds, src_col, tgt_col, split):
    cols = ds.column_names
    assert src_col in cols and tgt_col in cols, f"El split '{split}' debe contener columnas '{src_col}' y '{tgt_col}'. Columnas: {cols}"

for split in ["train", "validation", "test"]:
    assert split in raw_ds, f"Falta el split '{split}' en el dataset."
    _check_cols(raw_ds[split], cfg.src_col, cfg.tgt_col, split)

# pruebas rápidas
def take_fraction(ds, frac, seed=42):
    if frac >= 1.0:
        return ds
    n = max(1, int(len(ds) * frac))
    return ds.shuffle(seed=seed).select(range(n))

train_ds = take_fraction(raw_ds["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(raw_ds["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(raw_ds["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[:2])
print(f"Tam. train/val/test (fracción={cfg.fraction}):", len(train_ds), len(val_ds), len(test_ds))


DatasetDict({
    train: Dataset({
        features: ['zh', 'en'],
        num_rows: 417208
    })
    validation: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
    test: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
})
{'zh': ['第章 听到白小纯的话语，看到圣皇的迟疑，邪皇这里顿时呼吸一促，他目中刹那就露出凌厉之芒，右手猛的一挥，顿时那残破的红日，骤然幻化', '尤其是看到人群内的宋缺时，神算子立刻警惕，他当年在空城，是第一个跟随白小纯的，受到了重用，如今却成为了第二个，他顿时就视宋缺为竞争对手'], 'en': ['chapter- The Saint-Emperor hesitated, and the Vile-Emperor sucked in a breath, eyes flickering with cold light as he waved his hand to summon his damaged red sun', 'That was even more the case when he noticed Song Que among Bai Xiaochun’s men, which immediately got him even more on guard. Back in Sky City, Master God-Diviner had been the first to start following Bai Xiaochun again, and it had led to incredible benefits. Now, he was only the second to join him, which put Song Que in his sights as a major rival']}
Tam. train/val/test (fracción=1): 417208 52151 52151

## 4) Cargar tokenizador y modelo 

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.model_ckpt)


model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_ckpt)
model.to(device)

# Info útil
n_params = sum(p.numel() for p in model.parameters())
print(f"Modelo cargado: {cfg.model_ckpt}")
print(f"Parámetros totales: {n_params:,}")

Modelo cargado: Helsinki-NLP/opus-mt-zh-en
Parámetros totales: 77,943,296


## 5) Preprocesamiento y tokenización

In [7]:
# Función para tokenizar ejemplos
def preprocess_function(examples):
    # Tokenizar textos fuente y destino
    model_inputs = tokenizer(
        examples[cfg.src_col],
        max_length=cfg.max_source_length,
        padding=False,
        truncation=True
    )

    # Tokenizar target con "labels"
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples[cfg.tgt_col],
            max_length=cfg.max_target_length,
            padding=False,
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Aplicar tokenización a todo el dataset
tokenized_datasets = raw_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_ds["train"].column_names
)

train_ds = take_fraction(tokenized_datasets["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(tokenized_datasets["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(tokenized_datasets["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[0])


Map:   0%|          | 0/417208 [00:00<?, ? examples/s]

c:\Users\Usuario\miniconda3\envs\wuxia\Lib\site-packages\transformers\tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/52151 [00:00<?, ? examples/s]

Map:   0%|          | 0/52151 [00:00<?, ? examples/s]

{'input_ids': [440, 4137, 7, 5563, 4067, 1049, 17398, 2709, 3273, 2, 3213, 5251, 16968, 11, 11608, 11531, 2, 30189, 16968, 4258, 6275, 142, 14088, 333, 16683, 2, 182, 5659, 80, 52849, 33153, 9656, 854, 23153, 50521, 892, 27910, 2, 10284, 1211, 15834, 11, 333, 32430, 2, 6275, 142, 1095, 10973, 6875, 11, 5925, 75, 2, 54115, 7481, 19122, 1154, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [4101, 15, 24, 7567, 15, 339, 97, 6139, 672, 45659, 104, 2, 6, 3, 1612, 11241, 15, 339, 97, 6139, 672, 56637, 10, 12, 24174, 2, 6112, 58114, 34632, 27, 8706, 1407, 33, 169, 22800, 104, 174, 2239, 8, 52294, 174, 16168, 11386, 14243, 0]}


## 6) Evaluación 

In [9]:

from tqdm.auto import tqdm
import sacrebleu
from sacrebleu.metrics import CHRF, TER
from rouge_score import rouge_scorer
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import wordpunct_tokenize
import numpy as np
import torch
import time
import json
from bert_score import score as calc_bertscore
from comet import download_model, load_from_checkpoint
start = time.time()


# Descargar recursos de NLTK (para METEOR)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

#  Parámetros 
EVAL_MAX_SAMPLES = 1000        # None = todo el split
PRED_BEAMS = 4
BATCH_EVAL = max(1, cfg.batch_size // 2)


assert 'model' in globals(), "No se encontró `model`. Carga el modelo antes."
assert 'tokenizer' in globals(), "No se encontró `tokenizer`. Cárgalo antes."
assert 'val_ds' in globals() and 'test_ds' in globals(), "Faltan `val_ds` y/o `test_ds`."
assert 'cfg' in globals(), "Falta `cfg`."

# Asegurar pad_token_id
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id


#  Seleccionar split 
eval_raw = test_ds if len(test_ds) > 0 else val_ds
n_total = len(eval_raw)
n_eval = n_total if (EVAL_MAX_SAMPLES is None) else min(n_total, int(EVAL_MAX_SAMPLES))
assert n_eval > 0, "No hay ejemplos para evaluar."
def decode_ids_to_text(dataset, id_col):
    return [
        tokenizer.decode(ids, skip_special_tokens=True)
        for ids in dataset[id_col]
    ]

src_texts = decode_ids_to_text(eval_raw, "input_ids")[:n_eval]
ref_texts = decode_ids_to_text(eval_raw, "labels")[:n_eval]


#  Generación por lotes 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def batched_generate(texts, batch_size=8, max_length=128, num_beams=4):
    preds = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=cfg.max_source_length
            ).to(device)
            outputs = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=True
            )
            preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return preds

preds = batched_generate(
    src_texts,
    batch_size=BATCH_EVAL,
    max_length=cfg.max_target_length,
    num_beams=PRED_BEAMS
)

# #  Métricas 
# bleu_corpus = sacrebleu.corpus_bleu(preds, [ref_texts]).score

# chrf_metric = CHRF(word_order=2)
# chrf_corpus = chrf_metric.corpus_score(preds, [ref_texts]).score

# ter_metric = TER()
# ter_corpus = ter_metric.corpus_score(preds, [ref_texts]).score

# def compute_rougeL_f1(hyp_list, ref_list):
#     scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
#     f1s = []
#     for h, r in zip(hyp_list, ref_list):
#         s = scorer.score(r, h)
#         f1s.append(s['rougeL'].fmeasure)
#     return float(np.mean(f1s)) * 100.0
# rougeL_f1 = compute_rougeL_f1(preds, ref_texts)

# def compute_meteor(hyp_list, ref_list):
#     scores = []
#     for hyp, ref in zip(hyp_list, ref_list):
#         hyp_tok = wordpunct_tokenize(hyp) if isinstance(hyp, str) else hyp
#         ref_tok = wordpunct_tokenize(ref) if isinstance(ref, str) else ref
#         scores.append(meteor_score([ref_tok], hyp_tok))
#     return float(np.mean(scores)) * 100.0
# meteor_avg = compute_meteor(preds, ref_texts)


# --- BERTSCORE ---
print("Calculando BERTScore...")
# Extraemos el idioma destino desde cfg (si no lo tienes configurado así, cámbialo por "en", "es", etc.)
tgt_language = getattr(cfg, "tgt_lang", "multilingual")
_, _, F1 = calc_bertscore(preds, ref_texts, lang=tgt_language, verbose=True, rescale_with_baseline=True)
bertscore_avg = float(F1.mean()) * 100.0

# --- COMET ---
print("Calculando COMET (puede tardar un poco en cargar la primera vez)...")
comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)
comet_model.eval()

# COMET necesita el texto original (src), la predicción (mt) y la referencia (ref)
comet_data = [
    {"src": s, "mt": p, "ref": r} 
    for s, p, r in zip(src_texts, preds, ref_texts)
]
comet_output = comet_model.predict(comet_data, batch_size=BATCH_EVAL, progress_bar=True)
comet_avg = float(comet_output.system_score) * 100.0

end_time = time.time()

results = {
    "model" : getattr(cfg, "model_ckpt", "unknown_model"),
    "n_eval": n_eval,
    "num_beams": PRED_BEAMS,
    "batch_eval": BATCH_EVAL,
    # "sacrebleu": round(bleu_corpus, 4),
    # "chrf2": round(chrf_corpus, 4),
    # "ter": round(ter_corpus, 4),
    # "rougeL_f1": round(rougeL_f1, 4),
    # "meteor": round(meteor_avg, 4), 
    "bertscore": round(bertscore_avg, 4),
    "comet": round(comet_avg, 4),
    "execution_time": round(end_time - start, 2)
}

if hasattr(cfg, "evaluation_dir"):
    os.makedirs(cfg.evaluation_dir, exist_ok=True)
    res_file = os.path.join(cfg.evaluation_dir, getattr(cfg, "results_file", "results.json"))
    with open(res_file, "a", encoding="utf-8") as f:
        f.write("\n")
        f.write(json.dumps(results, ensure_ascii=False, indent=4))

print("\n--- RESULTADOS FINALES ---")
print(json.dumps(results, indent=4))


  0%|          | 0/125 [00:00<?, ?it/s]

Calculando BERTScore...
calculating scores...
computing bert embedding.


  0%|          | 0/32 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/16 [00:00<?, ?it/s]

done in 2.74 seconds, 365.27 sentences/sec
Calculando COMET (puede tardar un poco en cargar la primera vez)...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\Usuario\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\2760a223ac957f30acfb18c8aa649b01cf1d75f2\checkpoints\model.ckpt`
Encoder model frozen.
c:\Users\Usuario\miniconda3\envs\wuxia\Lib\site-packages\pytorch_lightning\core\saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to e


--- RESULTADOS FINALES ---
{
    "model": "Helsinki-NLP/opus-mt-zh-en",
    "n_eval": 1000,
    "num_beams": 4,
    "batch_eval": 8,
    "bertscore": 72.8423,
    "comet": 59.061,
    "execution_time": 124.15
}


In [ ]:
# results = {
#     "model" : cfg.model_ckpt,
#     "n_eval": n_eval,
#     "num_beams": PRED_BEAMS,
#     "batch_eval": BATCH_EVAL,
#     "sacrebleu": round(bleu_corpus, 4),
#     "chrf2": round(chrf_corpus, 4),
#     "ter": round(ter_corpus, 4),
#     "rougeL_f1": round(rougeL_f1, 4),
#     "meteor": round(meteor_avg, 4), 
#     "execution_time": round(end_time - start, 2)
# }

# os.makedirs(cfg.evaluation_dir, exist_ok=True)

# res_file = os.path.join(cfg.evaluation_dir, cfg.results_file)

# with open(res_file, "a", encoding="utf-8") as f:
#     f.write("\n")
#     f.write(json.dumps(results, ensure_ascii=False, indent=4))

# print(results)

{'model': 'Helsinki-NLP/opus-mt-zh-en', 'n_eval': 52151, 'num_beams': 4, 'batch_eval': 8, 'sacrebleu': 5.1229, 'chrf2': 23.5626, 'ter': 90.2934, 'rougeL_f1': 27.803, 'meteor': 23.7127, 'execution_time': 3515.09}


## 7) Muestra cualitativa (n ejemplos aleatorios)

In [11]:
import random
import torch

# Mostrar predicciones aleatorias 
n_show = 100
idxs = random.sample(range(len(eval_raw)), k=min(n_show, len(eval_raw)))

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i in idxs:
    # Decodificar texto fuente y referencia desde dataset tokenizado
    zh = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
    en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

    # Tokenizar entrada y mover a dispositivo
    inputs = tokenizer(zh, return_tensors="pt").to(device)

    # Generar traducción
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_length=cfg.max_target_length,
            num_beams=4,
            early_stopping=True
        )

    en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

    print("="*80)
    print("ZH:", zh)
    print("EN (ref):", en_ref)
    print("EN (pred):", en_pred)


ZH: 最关键的是——真正的天庭,不是在这里,而是在我们的心中
EN (ref): And most importantly, the true Heavenly Court is not here, it is within our hearts
EN (pred): The most important thing is that the real court, not here, is in our hearts.
ZH: 这一切他都清楚,可他还是脚步停顿下来,没有选择离去,因为 他不能错过这个机会,一旦错过,哪怕是在这炼魂壶的世界内,他想要去找到白浩,也都极为艰难,且一旦在这过程中,白浩的魂出现了灭亡,那么将是他一生的遗憾
EN (ref): He could not pass up this oppor tunity. After all, even knowing that Bai Hao’s soul was in the Necromancer Kettle wouldn’t necessarily do much good. Even going to search for him in that one particular area would be difficult. Furthermore, if Bai Hao’s soul were to die, then Bai Xiaochun would feel regret over the matter for the rest of his life
EN (pred): He knew that he had stopped and had no choice but to leave, because he couldn't miss the opportunity, and it was extremely difficult for him to find Baek Ho, even in this world of pots, and it would be his life's regret if his spirits died in the process.
ZH: 我不信
EN (ref): I do not believe it
EN (pred): I don't

In [14]:

from tqdm import tqdm

os.makedirs(cfg.translate_dir, exist_ok=True)
translate_path = os.path.join(cfg.translate_dir, cfg.translate_file)

with open(translate_path, "w", encoding="utf-8") as f:
    for i in tqdm(range(len(eval_raw) // 4)):
        zh = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
        en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

        inputs = tokenizer(zh, return_tensors="pt").to(device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_length=cfg.max_target_length,
                num_beams=4,
                early_stopping=True
            )

        en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

        # Guardar en el archivo
        f.write("="*80 + "\n")
        f.write("ZH: " + zh + "\n")
        f.write("EN (ref): " + en_ref + "\n")
        f.write("EN (pred): " + en_pred + "\n\n")


100%|██████████| 13037/13037 [47:10<00:00,  4.61it/s] 
